setup

In [1]:
import os
import pandas as pd
from qiime2 import Visualization
import matplotlib.pyplot as plt

%matplotlib inline

In [2]:
data_dir = '../data/raw'
results_dir = '../data/processed'

In [3]:
! ls

 Clustering.ipynb		  Setup.ipynb
'Quality Filtering.ipynb'	  denoise.ipynb
'Quality plot generation.ipynb'   gwendolin_data_summary.ipynb


import data

In [4]:
! ls -l $data_dir

total 695937
drwxr-xr-x 3 jovyan jovyan      4096 Oct  2 10:25 data
-rw-r--r-- 1 jovyan jovyan 712595535 Oct  2 10:31 fungut_forward_reads.qza
-rw-r--r-- 1 jovyan jovyan     18798 Oct  2 10:23 fungut_metadata.tsv


In [5]:
! echo 'Number of files in the directory is:' && ls $data_dir| wc -l

Number of files in the directory is:
3


In [6]:
!unzip -l ../data/raw/fungut_forward_reads.qza | grep fastq | head -10

  6218883  2024-09-10 14:52   3638611d-1767-413b-9390-70ee3d78e4ff/data/ERR5327540_01_L001_R1_001.fastq.gz
  2734350  2024-09-10 14:52   3638611d-1767-413b-9390-70ee3d78e4ff/data/ERR5327531_01_L001_R1_001.fastq.gz
  3907977  2024-09-10 14:52   3638611d-1767-413b-9390-70ee3d78e4ff/data/ERR5327266_01_L001_R1_001.fastq.gz
  5596947  2024-09-10 14:52   3638611d-1767-413b-9390-70ee3d78e4ff/data/ERR5327415_01_L001_R1_001.fastq.gz
  4274543  2024-09-10 14:52   3638611d-1767-413b-9390-70ee3d78e4ff/data/ERR5327489_01_L001_R1_001.fastq.gz
  1981961  2024-09-10 14:52   3638611d-1767-413b-9390-70ee3d78e4ff/data/ERR5327284_01_L001_R1_001.fastq.gz
  4617752  2024-09-10 14:52   3638611d-1767-413b-9390-70ee3d78e4ff/data/ERR5327464_01_L001_R1_001.fastq.gz
  4880282  2024-09-10 14:52   3638611d-1767-413b-9390-70ee3d78e4ff/data/ERR5327562_01_L001_R1_001.fastq.gz
  3780841  2024-09-10 14:52   3638611d-1767-413b-9390-70ee3d78e4ff/data/ERR5327513_01_L001_R1_001.fastq.gz
  4387898  2024-09-10 14:52   3638611

In [7]:
!unzip -l ../data/raw/fungut_forward_reads.qza | grep -c fastq.gz

150


In [8]:
metadata_df = pd.read_csv(f'{data_dir}/fungut_metadata.tsv', sep='\t', index_col=0)
metadata_df.sample(n=5)

,country_sample,state_sample,latitude_sample,longitude_sample,sex_sample,age_years_sample,height_cm_sample,weight_kg_sample,bmi_sample,diet_type_sample,ibd_sample,gluten_sample
ID,,,,,,,,,,,,
ERR5327515,USA,CA,37.3,-121.9,male,51.0,177.0,99.0,31.57,Omnivore,"Diagnosed by a medical professional (doctor, p...",No
ERR5327351,Australia,VIC,-38.1,145.5,female,10.0,144.0,34.0,16.4,Omnivore,I do not have this condition,No
ERR5327396,USA,CO,40.0,-105.2,male,36.0,180.0,63.0,19.6,Omnivore,I do not have this condition,I do not eat gluten because it makes me feel bad
ERR5327395,USA,CO,40.0,-105.2,male,36.0,180.0,63.0,19.6,Omnivore,I do not have this condition,I do not eat gluten because it makes me feel bad
ERR5327394,USA,FL,28.5,-81.4,male,65.0,172.0,81.0,27.37,Vegetarian but eat seafood,I do not have this condition,No


check qc

In [9]:
!qiime tools peek ../data/raw/fungut_forward_reads.qza

!qiime demux summarize \
  --i-data ../data/raw/fungut_forward_reads.qza \
  --o-visualization ../data/processed/demux_summary_gjx.qzv

UUID:        3638611d-1767-413b-9390-70ee3d78e4ff
Type:        SampleData[SequencesWithQuality]
Data format: SingleLanePerSampleSingleEndFastqDirFmt
Saved Visualization to: ../data/processed/demux_summary_gjx.qzv


In [10]:
Visualization.load(f"{results_dir}/demux_summary_gjx.qzv")

<visualization: Visualization uuid: 06a7f08c-b325-40cc-9cc2-e14766b0d322>

denoise

In [11]:
!qiime tools export --input-path ../data/raw/fungut_forward_reads.qza --output-path ../data/processed_ignored/exported/$(ls ../data/processed_ignored/exported | grep fastq.gz | head -1) | head -8
# not so sure should we cut primer？ how long？

ls: cannot access '../data/processed_ignored/exported': No such file or directory
Exported ../data/raw/fungut_forward_reads.qza as SingleLanePerSampleSingleEndFastqDirFmt to directory ../data/processed_ignored/exported/


In [12]:
%%bash
cd ../data/processed_ignored/exported
files=($(ls | grep fastq.gz))
for i in 0 49 99 149; do
  f=${files[$i]}
  echo "=== $f ==="
  zcat "$f" 2>/dev/null | awk 'NR%4==2' | head -10000 | cut -c1-28 | sort | uniq -c | sort -rn | head -3
done

=== ERR5327198_01_L001_R1_001.fastq.gz ===
   7961 GTAGGTGAACCTGCGGAAGGATCATTAA
    611 GTAGGTGAACCTGCGGAAGGATCATTAC
    299 GTAGGTGAACCTGCGGAAGGATCATTAG
=== ERR5327388_01_L001_R1_001.fastq.gz ===
   7401 GTAGGTGAACCTGCGGAAGGATCATTAC
   1903 GTAGGTGAACCTGCGGAAGGATCATTAT
     24 GTAGGTGAGCCTGCGGAAGGATCATTAC
=== ERR5327489_01_L001_R1_001.fastq.gz ===
   8198 GTAGGTGAACCTGCGGAAGGATCATTAG
   1137 GTAGGTGAACCTGCGGAAGGATCATTAC
     77 GTAGGTGAACCTGCGGAAGGATCATTAT
=== ERR5327620_01_L001_R1_001.fastq.gz ===
   9202 GTAGGTGAACCTGCGGAAGGATCATTAT
     90 GTAGGTGAACCTGCGGAAGGATCATTAC
     36 GTAGGTGAACCTGCGGAGGGATCATTAT


In [13]:
!qiime dada2 denoise-single \
  --i-demultiplexed-seqs ../data/raw/fungut_forward_reads.qza \
  --p-trim-left 27 \
  --p-trunc-len 0 \
  --p-n-threads 3 \
  --o-table ../data/processed_ignored/dada2_table.qza \
  --o-representative-sequences ../data/processed_ignored/dada2_rep_set.qza \
  --o-denoising-stats ../data/processed_ignored/dada2_stats.qza\
  --o-base-transition-stats ../data/processed_ignored/dada2_base_transition_stats.qza 

Saved FeatureTable[Frequency] to: ../data/processed_ignored/dada2_table.qza
Saved FeatureData[Sequence] to: ../data/processed_ignored/dada2_rep_set.qza
Saved SampleData[DADA2Stats] to: ../data/processed_ignored/dada2_stats.qza
Saved DADA2BaseTransitionStats to: ../data/processed_ignored/dada2_base_transition_stats.qza


In [14]:
!qiime metadata tabulate \
  --m-input-file ../data/processed_ignored/dada2_stats.qza \
  --o-visualization ../data/processed_ignored/dada2_stats.qzv

Saved Visualization to: ../data/processed_ignored/dada2_stats.qzv


In [15]:
from qiime2 import Visualization
Visualization.load("../data/processed_ignored/dada2_stats.qzv")

<visualization: Visualization uuid: f30d8eac-2267-4195-983c-52f1b9374b2a>

In [16]:
!qiime feature-table tabulate-seqs \
  --i-data ../data/processed_ignored/dada2_rep_set.qza \
  --o-visualization ../data/processed_ignored/dada2_rep_set.qzv

Saved Visualization to: ../data/processed_ignored/dada2_rep_set.qzv


In [17]:
Visualization.load("../data/processed_ignored/dada2_rep_set.qzv")

<visualization: Visualization uuid: 7959ab0d-a8fd-4e5e-872a-f9cef6b2221c>

In [18]:
!qiime feature-table summarize \
  --i-table ../data/processed_ignored/dada2_table.qza \
  --m-metadata-file ../data/raw/fungut_metadata.tsv \
  --o-feature-frequencies ../data/processed_ignored/dada2_table_feature_frequencies.qza \
  --o-sample-frequencies ../data/processed_ignored/dada2_table_sample_frequencies.qza \
  --o-summary ../data/processed/dada2_table.qzv

Saved ImmutableMetadata to: ../data/processed_ignored/dada2_table_feature_frequencies.qza
Saved ImmutableMetadata to: ../data/processed_ignored/dada2_table_sample_frequencies.qza
Saved Visualization to: ../data/processed/dada2_table.qzv


In [19]:
Visualization.load("../data/processed//dada2_table.qzv")

<visualization: Visualization uuid: ae8ee48e-9a1b-41a0-82bd-c7bdd92d8d75>

In [20]:
!mv ../data/processed_ignored/exported ../data/processed_ignored/exported

mv: cannot move '../data/processed_ignored/exported' to a subdirectory of itself, '../data/processed_ignored/exported/exported'


In [21]:
!ls ../data
!ls ../data/processed_ignored/exported | wc -l

processed  processed_ignored  raw
152
